In [5]:
from pyspark.sql import functions as F


StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, 7, Finished, Available, Finished, False)

In [6]:
df = spark.read.table("dbo.delivery_logistics")
display(df.limit(10))

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 219c14ba-16b4-4492-aa83-5008daf8bc4e)

In [7]:
# Rename columns to remove spaces (fixes the Delta error)
new_columns = [col.replace(" ", "_") for col in df.columns]
bronze_df = df.toDF(*new_columns)

bronze_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_delivery_logistics")

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, 9, Finished, Available, Finished, False)

In [8]:
from pyspark.sql import functions as F, types as T

# 1. Read from Bronze
bronze_df = spark.read.table("bronze_delivery_logistics")

# 2. Remove exact duplicate rows (identical across every column)
silver_df = bronze_df.dropDuplicates()

# 3. Trim extra spaces from all string columns
string_cols = [f.name for f in silver_df.schema.fields if str(f.dataType) == "StringType()"]
for c in string_cols:
    silver_df = silver_df.withColumn(c, F.trim(F.col(c)))

# 4. Cast columns to their correct data type where they aren't already
numeric_cols = ["distance_km", "package_weight_kg", "delivery_cost"]
for c in numeric_cols:
    silver_df = silver_df.withColumn(c, F.col(c).cast(T.DoubleType()))

silver_df = silver_df.withColumn("delivery_rating", F.col("delivery_rating").cast(T.IntegerType()))

display(silver_df)

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 091b1422-5576-4c5c-8a17-bfe2bfc35e85)

In [9]:
from pyspark.sql import functions as F

# 1. Count nulls in every column of silver_df
null_counts = silver_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in silver_df.columns
])
display(null_counts)

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 135d1ffb-2a27-4047-bdc3-742cddf2ab6a)

In [10]:
# 5. Check row count after cleaning
print(f"Bronze row count: {bronze_df.count()}")
print(f"Silver row count: {silver_df.count()}")

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, 12, Finished, Available, Finished, False)

Bronze row count: 25000
Silver row count: 25000


In [11]:
# Step 8: Write to silver Delta table

silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_delivery_logistics")

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, 13, Finished, Available, Finished, False)

AnalysisException: [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'delivery_time_hours' and 'delivery_time_hours'

In [ ]:
# Step 1: Read from the Silver Delta table
silver_df = spark.read.table("silver_delivery_logistics")


StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql import functions as F

# Step 2: Build dimension tables using natural keys
# Each dimension = the distinct values of one categorical column.
# No new key is generated — the text value itself (e.g. "dhl", "west")
# IS the key of that dimension table.

dim_partner       = silver_df.select("delivery_partner").distinct()
dim_package_type  = silver_df.select("package_type").distinct()
dim_vehicle_type  = silver_df.select("vehicle_type").distinct()
dim_delivery_mode = silver_df.select("delivery_mode").distinct()
dim_region        = silver_df.select("region").distinct()
dim_weather       = silver_df.select("weather_condition").distinct()

display(dim_partner)

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, -1, Cancelled, , Cancelled, True)

In [ ]:
# Step 3: Sanity check — row counts should match the number of distinct values
print("dim_partner:", dim_partner.count())              # expect 9
print("dim_package_type:", dim_package_type.count())    # expect 9
print("dim_vehicle_type:", dim_vehicle_type.count())    # expect 6
print("dim_delivery_mode:", dim_delivery_mode.count())  # expect 4
print("dim_region:", dim_region.count())                # expect 5
print("dim_weather:", dim_weather.count())               # expect 6

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, -1, Cancelled, , Cancelled, True)

In [ ]:
# Step 4: Build the fact table
# No joins needed here — silver_df already stores the natural key text values
# (delivery_partner, region, etc.) directly, and those same values are the
# key of each dimension table above. The relationship exists simply because
# the values match — nothing to look up or generate.

fact_delivery = silver_df.withColumn(
    "time_variance_hours", F.col("delivery_time_hours") - F.col("expected_time_hours")
).select(
    "delivery_id",
    "delivery_partner", "package_type", "vehicle_type",
    "delivery_mode", "region", "weather_condition",
    "distance_km", "package_weight_kg",
    "delivery_time_hours", "expected_time_hours", "time_variance_hours",
    "delayed", "delivery_status", "delivery_rating", "delivery_cost",
)

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, -1, Cancelled, , Cancelled, True)

In [ ]:
# Step 5: Verify no rows were lost
print("Silver row count:", silver_df.count())
print("Fact row count:", fact_delivery.count())   # should both be 25000



StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, -1, Cancelled, , Cancelled, True)

In [ ]:
# Step 6: Write dimension and fact tables to the Gold layer
dim_partner.write.mode("overwrite").format("delta").saveAsTable("gold_dim_partner")
dim_package_type.write.mode("overwrite").format("delta").saveAsTable("gold_dim_package_type")
dim_vehicle_type.write.mode("overwrite").format("delta").saveAsTable("gold_dim_vehicle_type")
dim_delivery_mode.write.mode("overwrite").format("delta").saveAsTable("gold_dim_delivery_mode")
dim_region.write.mode("overwrite").format("delta").saveAsTable("gold_dim_region")
dim_weather.write.mode("overwrite").format("delta").saveAsTable("gold_dim_weather")

fact_delivery.write.mode("overwrite").format("delta").saveAsTable("gold_fact_delivery")

print("Gold layer written (natural-key version): 6 dimension tables + 1 fact table")

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql import functions as F, types as T

raw_path = "Files/Delivery_Logistics.csv"   # <-- adjust if your file path differs

# Read raw CSV as plain text (no timestamp auto-parsing this time)
raw = spark.read.option("header", True).csv(raw_path)

# 1. Remove exact duplicate rows
silver_df = raw.dropDuplicates()

# 2. Trim string columns
string_cols = [c for c, t in silver_df.dtypes if t == "string"]
for c in string_cols:
    silver_df = silver_df.withColumn(c, F.trim(F.col(c)))

# 3. Cast numeric columns
silver_df = (
    silver_df
    .withColumn("delivery_id", F.col("delivery_id").cast(T.DoubleType()))
    .withColumn("distance_km", F.col("distance_km").cast(T.DoubleType()))
    .withColumn("package_weight_kg", F.col("package_weight_kg").cast(T.DoubleType()))
    .withColumn("delivery_cost", F.col("delivery_cost").cast(T.DoubleType()))
    .withColumn("delivery_rating", F.col("delivery_rating").cast(T.IntegerType()))
)

# 4. Fix the time columns by pulling the real hour value straight out of the
# text (avoids Spark ever treating them as timestamps, which loses precision)
silver_df = (
    silver_df
    .withColumn("delivery_time_hours", F.regexp_extract("delivery_time_hours", r"\.(\d+)$", 1).cast(T.LongType()))
    .withColumn("expected_time_hours", F.regexp_extract("expected_time_hours", r"\.(\d+)$", 1).cast(T.LongType()))
)

print("Silver row count:", silver_df.count())  # expect 25000
silver_df.write.mode("overwrite").format("delta").saveAsTable("silver_delivery_logistics")

# Rebuild Gold
dim_partner       = silver_df.select("delivery_partner").distinct()
dim_package_type  = silver_df.select("package_type").distinct()
dim_vehicle_type  = silver_df.select("vehicle_type").distinct()
dim_delivery_mode = silver_df.select("delivery_mode").distinct()
dim_region        = silver_df.select("region").distinct()
dim_weather       = silver_df.select("weather_condition").distinct()

fact_delivery = silver_df.withColumn(
    "time_variance_hours", F.col("delivery_time_hours") - F.col("expected_time_hours")
).select(
    "delivery_id", "delivery_partner", "package_type", "vehicle_type",
    "delivery_mode", "region", "weather_condition",
    "distance_km", "package_weight_kg",
    "delivery_time_hours", "expected_time_hours", "time_variance_hours",
    "delayed", "delivery_status", "delivery_rating", "delivery_cost",
)

print("Fact row count:", fact_delivery.count())  # expect 25000

dim_partner.write.mode("overwrite").format("delta").saveAsTable("gold_dim_partner")
dim_package_type.write.mode("overwrite").format("delta").saveAsTable("gold_dim_package_type")
dim_vehicle_type.write.mode("overwrite").format("delta").saveAsTable("gold_dim_vehicle_type")
dim_delivery_mode.write.mode("overwrite").format("delta").saveAsTable("gold_dim_delivery_mode")
dim_region.write.mode("overwrite").format("delta").saveAsTable("gold_dim_region")
dim_weather.write.mode("overwrite").format("delta").saveAsTable("gold_dim_weather")
fact_delivery.write.mode("overwrite").format("delta").saveAsTable("gold_fact_delivery")

print("Gold layer written successfully.")

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, -1, Cancelled, , Cancelled, True)

In [ ]:
silver_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_delivery_logistics")

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, -1, Cancelled, , Cancelled, True)

In [ ]:
fact_delivery.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_fact_delivery")

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, -1, Cancelled, , Cancelled, True)

In [12]:
from pyspark.sql import functions as F, types as T

df = spark.read.table("dbo.delivery_logistics")
display(df.limit(10))

string_cols = [c for c, t in silver_df.dtypes if t == "string"]
for c in string_cols:
    silver_df = silver_df.withColumn(c, F.trim(F.col(c)))

silver_df = (
    silver_df
    .withColumn("delivery_id", F.col("delivery_id").cast(T.DoubleType()))
    .withColumn("distance_km", F.col("distance_km").cast(T.DoubleType()))
    .withColumn("package_weight_kg", F.col("package_weight_kg").cast(T.DoubleType()))
    .withColumn("delivery_cost", F.col("delivery_cost").cast(T.DoubleType()))
    .withColumn("delivery_rating", F.col("delivery_rating").cast(T.IntegerType()))
    .withColumn("delivery_time_hours", F.regexp_extract("delivery_time_hours", r"\.(\d+)$", 1).cast(T.LongType()))
    .withColumn("expected_time_hours", F.regexp_extract("expected_time_hours", r"\.(\d+)$", 1).cast(T.LongType()))
)

silver_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_delivery_logistics")

dim_partner       = silver_df.select("delivery_partner").distinct()
dim_package_type  = silver_df.select("package_type").distinct()
dim_vehicle_type  = silver_df.select("vehicle_type").distinct()
dim_delivery_mode = silver_df.select("delivery_mode").distinct()
dim_region        = silver_df.select("region").distinct()
dim_weather       = silver_df.select("weather_condition").distinct()

fact_delivery = silver_df.withColumn(
    "time_variance_hours", F.col("delivery_time_hours") - F.col("expected_time_hours")
).select(
    "delivery_id", "delivery_partner", "package_type", "vehicle_type",
    "delivery_mode", "region", "weather_condition",
    "distance_km", "package_weight_kg",
    "delivery_time_hours", "expected_time_hours", "time_variance_hours",
    "delayed", "delivery_status", "delivery_rating", "delivery_cost",
)

print("Silver row count:", silver_df.count())
print("Fact row count:", fact_delivery.count())

dim_partner.write.mode("overwrite").format("delta").saveAsTable("gold_dim_partner")
dim_package_type.write.mode("overwrite").format("delta").saveAsTable("gold_dim_package_type")
dim_vehicle_type.write.mode("overwrite").format("delta").saveAsTable("gold_dim_vehicle_type")
dim_delivery_mode.write.mode("overwrite").format("delta").saveAsTable("gold_dim_delivery_mode")
dim_region.write.mode("overwrite").format("delta").saveAsTable("gold_dim_region")
dim_weather.write.mode("overwrite").format("delta").saveAsTable("gold_dim_weather")
fact_delivery.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_fact_delivery")

print("Gold layer written successfully.")

StatementMeta(, 63f8ae6b-5440-4785-9281-32d37b848047, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b8beead8-1ae4-430a-b59d-9bd5e15f5dba)

Silver row count: 25000
Fact row count: 25000
Gold layer written successfully.
